<a href="https://colab.research.google.com/github/Alex-Sun-316/Sudoku/blob/main/Sudoku_Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json

def getel(s):
    assert len(s) == 1
    return next(iter(s))


class Sudoku(object):
    def __init__(self, elements):
        if isinstance(elements, Sudoku):
            self.m = [[x.copy() for x in row] for row in elements.m]
        else:
            assert len(elements) == 9
            for s in elements:
                assert len(s) == 9

            self.m = []
            for s in elements:
                row = []
                for c in s:
                    if isinstance(c, str):
                        if c.isdigit():
                            row.append({int(c)})
                        else:
                            row.append({1, 2, 3, 4, 5, 6, 7, 8, 9})
                    else:
                        assert isinstance(c, set)
                        row.append(c)
                self.m.append(row)

    def show(self, details=False):
        if details:
            print("+-----------------------------+-----------------------------+-----------------------------+")
            for i in range(9):
                r = '|'
                for j in range(9):
                    s = ''
                    for k in range(1, 10):
                        s += str(k) if k in self.m[i][j] else '_'
                    r += s
                    r += '|' if (j + 1) % 3 == 0 else ' '
                print(r)
                if (i + 1) % 3 == 0:
                    print("+-----------------------------+-----------------------------+-----------------------------+")
        else:
            print("+---+---+---+")
            for i in range(9):
                r = '|'
                for j in range(9):
                    if len(self.m[i][j]) == 1:
                        r += str(getel(self.m[i][j]))
                    else:
                        r += "."
                    if (j + 1) % 3 == 0:
                        r += "|"
                print(r)
                if (i + 1) % 3 == 0:
                    print("+---+---+---+")

    def to_string(self):
        as_lists = [[list(self.m[i][j]) for j in range(9)] for i in range(9)]
        return json.dumps(as_lists)

    @staticmethod
    def from_string(s):
        as_lists = json.loads(s)
        as_sets = [[set(el) for el in row] for row in as_lists]
        return Sudoku(as_sets)

    def __eq__(self, other):
        return self.m == other.m

In [2]:
class Unsolvable(Exception):
    pass


def sudoku_ruleout(self, i, j, x):
    c = self.m[i][j]
    n = len(c)
    c.discard(x)
    self.m[i][j] = c

    if len(c) == 0:
        raise Unsolvable()

    if len(c) == 1 and n > 1:
        return {(i, j)}
    return set()


Sudoku.ruleout = sudoku_ruleout

In [3]:
def sudoku_propagate_cell(self, ij):
    i, j = ij

    if len(self.m[i][j]) > 1:
        return set()

    newly_singleton = set()
    x = getel(self.m[i][j])

    # same row
    for jj in range(9):
        if jj != j:
            newly_singleton.update(self.ruleout(i, jj, x))

    # same column
    for ii in range(9):
        if ii != i:
            newly_singleton.update(self.ruleout(ii, j, x))

    # same 3x3 block
    bi = (i // 3) * 3
    bj = (j // 3) * 3
    for ii in range(bi, bi + 3):
        for jj in range(bj, bj + 3):
            if (ii, jj) != (i, j):
                newly_singleton.update(self.ruleout(ii, jj, x))

    return newly_singleton


Sudoku.propagate_cell = sudoku_propagate_cell

In [4]:
def sudoku_propagate_all_cells_once(self):
    for i in range(9):
        for j in range(9):
            self.propagate_cell((i, j))


Sudoku.propagate_all_cells_once = sudoku_propagate_all_cells_once

In [5]:
def sudoku_full_propagation(self, to_propagate=None):
    if to_propagate is None:
        to_propagate = {(i, j) for i in range(9) for j in range(9)}

    while len(to_propagate) > 0:
        cell = to_propagate.pop()
        to_propagate.update(self.propagate_cell(cell))


Sudoku.full_propagation = sudoku_full_propagation

In [6]:
sd = Sudoku([
    '53__7____',
    '6__195___',
    '_98____6_',
    '8___6___3',
    '4__8_3__1',
    '7___2___6',
    '_6____28_',
    '___419__5',
    '____8__79'
])

sd.show()
sd.full_propagation()
sd.show(details=True)

+---+---+---+
|53.|.7.|...|
|6..|195|...|
|.98|...|.6.|
+---+---+---+
|8..|.6.|..3|
|4..|8.3|..1|
|7..|.2.|..6|
+---+---+---+
|.6.|...|28.|
|...|419|..5|
|...|.8.|.79|
+---+---+---+
+-----------------------------+-----------------------------+-----------------------------+
|____5____ __3______ ___4_____|_____6___ ______7__ _______8_|________9 1________ _2_______|
|_____6___ ______7__ _2_______|1________ ________9 ____5____|__3______ ___4_____ _______8_|
|1________ ________9 _______8_|__3______ ___4_____ _2_______|____5____ _____6___ ______7__|
+-----------------------------+-----------------------------+-----------------------------+
|_______8_ ____5____ ________9|______7__ _____6___ 1________|___4_____ _2_______ __3______|
|___4_____ _2_______ _____6___|_______8_ ____5____ __3______|______7__ ________9 1________|
|______7__ 1________ __3______|________9 _2_______ ___4_____|_______8_ ____5____ _____6___|
+-----------------------------+-----------------------------+---------------------